# Imports and Setup

In [6]:
import os
import json
import ast
from openai import OpenAI

openai_key_file = "/Users/jinjinzhao/Documents/work_projects/my_keys/my_keys/openai_jinjin.key"
with open(openai_key_file, 'r') as f:
    openai_key = f.read()

os.environ["OPENAI_API_KEY"] = openai_key
client = OpenAI()
MODEL = "gpt-5.4"

# Generate Data Science Tasks
Ask an openai model to brainstorm list of descriptions of n AI tasks that can be evaluated with an OpenAI API call and HuggingFace API. Restrict the API to tasks with up to thousands of samples. Return a list of task descriptions as a python list of strings. For example, describing tasks sentiment analysis over ag_news.

In [10]:
def generate_data_science_tasks(n: int = 10) -> list:
    prompt = f"""Brainstorm a list of {n} descriptions of AI tasks that can be evaluated using the OpenAI API and HuggingFace datasets.
Restrict to tasks with datasets that have up to a few thousand samples (not millions).
Each description should specify both the task type and the dataset,
Return ONLY a valid Python list of strings, no explanation."""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
    )
    raw = response.choices[0].message.content.strip()
    return ast.literal_eval(raw)


tasks = generate_data_science_tasks(10)
tasks

['Sentiment analysis on the SST-2 dataset from the GLUE benchmark, classifying movie review sentences as positive or negative.',
 'Natural language inference on the SNLI dataset, determining whether a premise entails, contradicts, or is neutral with respect to a hypothesis.',
 'Paraphrase detection on the MRPC dataset from GLUE, identifying whether two sentences have the same meaning.',
 'Question answering on the SQuAD v1.1 subset, extracting the answer span from a passage for a given question.',
 'Summarization on the XSum dataset using a small evaluation subset, generating a one-sentence summary of a news article.',
 'Text classification for emotion recognition on the Emotion dataset, predicting labels such as joy, sadness, anger, and fear from short texts.',
 'Toxic comment classification on the civil_comments dataset using a small sampled subset, predicting whether a comment is toxic.',
 'Commonsense reasoning on the COPA dataset, choosing the more plausible cause or effect for a 

# Initial Notebook Generation
Given an initial ai task as a string, use OpenAI API to generate a jupyter notebook for a simple workflow of that task. Assume you can make OpenAI and HuggingFace API calls. Allow us to parameterize location to store the jupyter notebook, the notebook name, and the task. Allow a reasonable simple OpenAI workflow. Eg. Plan then create.

In [9]:
def generate_notebook(task: str, notebook_dir: str, notebook_name: str) -> str:
    # Step 1: Plan
    plan_response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": f"""You are an expert data scientist. Plan a simple Jupyter notebook workflow for this AI task:

Task: {task}

Assume access to OpenAI API and HuggingFace datasets.
Write a concise step-by-step plan (5-7 steps) for the notebook."""}],
    )
    plan = plan_response.choices[0].message.content.strip()

    # Step 2: Generate notebook JSON
    nb_response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": f"""You are an expert data scientist. Generate a complete Jupyter notebook as valid JSON for this AI task.

Task: {task}

Plan:
{plan}

Requirements:
- Use HuggingFace datasets to load data
- Use OpenAI API for model inference
- Include cells for imports, data loading, inference, and evaluation
- The notebook must be valid .ipynb JSON (nbformat 4)
- Return ONLY the raw JSON, no markdown fences or explanation."""}],
    )
    raw = nb_response.choices[0].message.content.strip()
    if raw.startswith("```"):
        raw = raw.split("```", 2)[1]
        if raw.startswith("json"):
            raw = raw[4:]
        raw = raw.rsplit("```", 1)[0].strip()

    nb = json.loads(raw)
    os.makedirs(notebook_dir, exist_ok=True)
    path = os.path.join(notebook_dir, notebook_name)
    with open(path, "w") as f:
        json.dump(nb, f, indent=1)
    print(f"Notebook saved to {path}")
    return path

In [11]:
notebook_dir = "../notebooks/test_batch"
notebook_name = "test.ipynb"

In [12]:
generate_notebook(tasks[0], notebook_dir, notebook_name)

Notebook saved to ../notebooks/test_batch/test.ipynb


'../notebooks/test_batch/test.ipynb'

# Plan Variations of Notebook Task
A function when given an initial jupyter notebook, use OpenAI API to generate list of descriptions of n variations to the notebook. Return a list of task descriptions as a python list of strings.

In [13]:
def plan_notebook_variations(notebook_path: str, n: int = 5) -> list:
    with open(notebook_path) as f:
        nb = json.load(f)

    cells_text = [
        "".join(cell.get("source", []))
        for cell in nb.get("cells", [])
        if "".join(cell.get("source", [])).strip()
    ]
    notebook_text = "\n\n".join(cells_text[:20])

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": f"""Here is a Jupyter notebook:

{notebook_text}

Generate {n} distinct variation descriptions for this notebook. Each variation should meaningfully change at least aspect such as the dataset, prompting strategy, model, or evaluation method.
Return ONLY a valid Python list of {n} strings, each describing one variation. No explanation."""}],
    )
    raw = response.choices[0].message.content.strip()
    return ast.literal_eval(raw)

In [15]:
notebook_path = "../notebooks/test_batch/test.ipynb"
variations = plan_notebook_variations(notebook_path, n = 5)
variations

['Replace SST-2 with the IMDb movie reviews dataset and compare zero-shot sentiment classification performance on longer full-review texts versus the original sentence-level setup.',
 'Change the prompting strategy from zero-shot to few-shot by adding several labeled positive and negative movie-review examples in the prompt, then evaluate whether accuracy and F1 improve on the same validation subset.',
 'Run the notebook as a model comparison study by evaluating multiple OpenAI models such as gpt-4o-mini and gpt-4.1-mini on the same sampled subset, reporting accuracy, confusion matrices, latency, and cost estimates.',
 'Modify the task from binary sentiment classification to confidence-aware classification by asking the model to return both a label and a confidence score in JSON format, then analyze calibration and performance at different confidence thresholds.',
 'Replace the balanced random subset evaluation with a robustness benchmark that tests prompt sensitivity by using several 

# Specify Variation Parameters
Given the list of variation descriptions, use OpenAI API to produce a structured specification for each variation. Each item in the output is a dictionary with a `variation_description` (str) and a `specification` (list of parameter dicts, each with `parameter`, `min`, `max`, and `values` — a randomly-sized sample of concrete values between min and max). Takes the variations list and a max number of specs per parameter as input.

In [ ]:
import random

def specify_variations(variations: list, lo: int = 2, hi: int = 4) -> list:
    """
    For each variation description, ask OpenAI to identify tunable parameters
    with sensible value ranges. Then randomly sample between lo and hi concrete
    values per parameter (re-randomized independently for each variation).

    Args:
        variations: list of variation description strings
        lo: minimum number of concrete values to sample per parameter
        hi: maximum number of concrete values to sample per parameter

    Returns a list of dicts:
        {
            "variation_description": str,
            "specification": [
                {"parameter": str, "min": number, "max": number, "values": list},
                ...
            ]
        }
    """
    prompt_template = """You are given a description of a notebook variation:

"{variation}"

Identify all tunable parameters in this variation (e.g. num_examples, temperature, max_tokens, dataset_split_size, etc.).
For each parameter, determine a sensible numeric min and max.

Return ONLY a valid Python list of dicts, one per parameter, like:
[
  {{"parameter": "num_examples", "min": 1, "max": 10}},
  {{"parameter": "temperature", "min": 0.0, "max": 1.0}}
]
No explanation."""

    results = []
    for variation in variations:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt_template.format(variation=variation)}],
        )
        raw = response.choices[0].message.content.strip()
        params = ast.literal_eval(raw)

        specification = []
        for p in params:
            p_min, p_max = p["min"], p["max"]
            # Randomize number of values independently for each variation
            k = random.randint(lo, hi)
            if isinstance(p_min, float) or isinstance(p_max, float):
                values = sorted(round(random.uniform(p_min, p_max), 2) for _ in range(k))
            else:
                population = list(range(int(p_min), int(p_max) + 1))
                k = min(k, len(population))
                values = sorted(random.sample(population, k))
            specification.append({
                "parameter": p["parameter"],
                "min": p_min,
                "max": p_max,
                "values": values,
            })

        results.append({
            "variation_description": variation,
            "specification": specification,
        })

    return results


specified_variations = specify_variations(variations, lo=2, hi=4)
specified_variations

# Generate Notebook Variation
A function when given a notebook file location and a variation string, a new file name, generates a new variation of the notebook in the same location with the file name.

In [ ]:
def generate_notebook_variation(
    notebook_name: str,
    variation: str,
    specification: list,
    source_notebook_path: str,
    output_dir: str,
) -> list:
    """
    For each spec in specification, generates one notebook implementing the
    variation with that specific parameter fixed to its sampled values.
    Saves files as {notebook_name}_{i}.ipynb in output_dir.
    Returns a list of saved file paths.
    """
    with open(source_notebook_path) as f:
        nb_json = f.read()

    os.makedirs(output_dir, exist_ok=True)
    saved_paths = []

    for i, spec in enumerate(specification):
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": f"""Here is a Jupyter notebook in JSON format:

{nb_json}

Apply this variation: {variation}

Specifically, parameterize the notebook for:
  parameter: {spec['parameter']}
  values to use: {spec['values']}  (range: {spec['min']} to {spec['max']})

The notebook should be configured to run with these specific values for {spec['parameter']}.
Return ONLY the valid .ipynb JSON for the modified notebook. No explanation or markdown fences."""}],
        )
        raw = response.choices[0].message.content.strip()
        if raw.startswith("```"):
            raw = raw.split("```", 2)[1]
            if raw.startswith("json"):
                raw = raw[4:]
            raw = raw.rsplit("```", 1)[0].strip()

        nb = json.loads(raw)
        file_name = f"{notebook_name}_{i}.ipynb"
        save_path = os.path.join(output_dir, file_name)
        with open(save_path, "w") as f:
            json.dump(nb, f, indent=1)
        print(f"Saved: {save_path}")
        saved_paths.append(save_path)

    return saved_paths

# Given notebook ask for general descriptions
Given a notebook file location and a task string, call the openai api to complete the task based on the notebook and return output. 


In [ ]:
def query_notebook(notebook_path: str, task: str) -> str:
    with open(notebook_path) as f:
        nb = json.load(f)

    cells_text = [
        "".join(cell.get("source", []))
        for cell in nb.get("cells", [])
        if "".join(cell.get("source", [])).strip()
    ]
    notebook_text = "\n\n".join(cells_text[:20])

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": f"""Here is a Jupyter notebook:

{notebook_text}

Task: {task}

Complete the task based on the notebook content and return the output."""}],
    )
    return response.choices[0].message.content.strip()